[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umangbagadi03/build-your-own-coding-agent/blob/main/notebooks/06_precise_edits.ipynb)

# 06. Precise File Edits

**Part 2: Tools** · about 25 minutes

The agent from notebook 05 fixes code by rewriting the whole file. On a 13-line file nobody notices. On a real project it's a disaster waiting to happen:

- To change one character, the model has to type out every other character too, and output tokens are the slow, expensive kind.
- Any slip while copying silently changes code it never meant to touch. It could drop a function, mangle a comment, or stop halfway when it reaches its output limit.

So every serious coding agent edits by **search and replace**. The model says "find this exact text and put this in its place", and your code makes the change. This notebook builds that tool, and also the boring details that decide whether it works in practice: line numbers, ranges, and error messages a model can recover from.

**By the end you will have:**

- `read_file` that shows line numbers and can read just a range
- `find`, so the agent can jump to the right place instead of reading everything
- `edit_file`, with the one rule that makes search-and-replace safe
- measured numbers on how much less the model has to write

In [1]:
import pathlib, urllib.request

if not pathlib.Path("llm_setup.py").exists():  # Colab opens one notebook at a time, so fetch the helper
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/umangbagadi03/build-your-own-coding-agent/main/notebooks/llm_setup.py", "llm_setup.py"
    )

from llm_setup import connect

llm = connect()

OFFLINE MODE: a scripted local server, not a real model.
The HTTP calls are real. The answers are canned for these lessons.
Token counts are rough estimates and timings are simulated.
Set GROQ_API_KEY (free tier) or LLM_PROVIDER=ollama to use a real model.


In [2]:
import difflib
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import requests

url = f"{llm.base_url}/chat/completions"

headers = {"Content-Type": "application/json"}
if llm.api_key:
    headers["Authorization"] = f"Bearer {llm.api_key}"

## 1. A slightly bigger project

A small inventory module for a corner shop. It's still short, but long enough that rewriting it to fix one line starts to look silly. One function has a bug, and two tests catch it.

In [3]:
INVENTORY = '''\
"""Small inventory helpers for a corner shop."""

import json


def load_items(path):
    """Read items from a JSON file."""
    with open(path) as f:
        return json.load(f)


def find_item(items, name):
    """Return the item with this name, or None."""
    for item in items:
        if item["name"].lower() == name.lower():
            return item
    return None


def needs_restock(item):
    """True when stock has fallen to the reorder level or below."""
    return item["stock"] < item["reorder_level"]


def total_value(items):
    """Value of everything on the shelves."""
    return sum(item["stock"] * item["price"] for item in items)


def low_stock(items):
    """Names of items that need restocking, alphabetically."""
    return sorted(item["name"] for item in items if needs_restock(item))


def apply_sale(item, percent):
    """Return a copy of the item with its price reduced by percent."""
    sale = dict(item)
    sale["price"] = round(item["price"] * (1 - percent / 100), 2)
    return sale


def format_line(item):
    """One line of the stock report."""
    flag = "  RESTOCK" if needs_restock(item) else ""
    return f"{item['name']:<12}{item['stock']:>5}{flag}"


def stock_report(items):
    """The full report, one item per line."""
    return "\\n".join(format_line(item) for item in sorted(items, key=lambda i: i["name"]))
'''

TESTS = '''\
import unittest

from inventory import apply_sale, find_item, low_stock, needs_restock, total_value

ITEMS = [
    {"name": "Rice", "stock": 3, "reorder_level": 5, "price": 60.0},
    {"name": "Tea", "stock": 5, "reorder_level": 5, "price": 120.0},
    {"name": "Sugar", "stock": 9, "reorder_level": 4, "price": 45.0},
]


class InventoryTests(unittest.TestCase):
    def test_restock_below_level(self):
        self.assertTrue(needs_restock(ITEMS[0]))

    def test_restock_at_exact_level(self):
        self.assertTrue(needs_restock(ITEMS[1]))

    def test_low_stock_names(self):
        self.assertEqual(low_stock(ITEMS), ["Rice", "Tea"])

    def test_total_value(self):
        self.assertEqual(total_value(ITEMS), 3 * 60 + 5 * 120 + 9 * 45)

    def test_find_is_case_insensitive(self):
        self.assertEqual(find_item(ITEMS, "tea")["name"], "Tea")

    def test_sale(self):
        self.assertEqual(apply_sale(ITEMS[2], 10)["price"], 40.5)


if __name__ == "__main__":
    unittest.main()
'''

workspace = Path(tempfile.mkdtemp(prefix="agent-workspace-")).resolve()
(workspace / "inventory.py").write_text(INVENTORY)
(workspace / "test_inventory.py").write_text(TESTS)

print(f"inventory.py: {len(INVENTORY.splitlines())} lines, {len(INVENTORY):,} characters")

inventory.py: 50 lines, 1,360 characters


## 2. What a one-character fix costs

The bug is `<` where it should be `<=`. Here are the two ways an agent could ask for that fix, measured by what the model has to *write*.

In [4]:
fixed_source = INVENTORY.replace(
    'return item["stock"] < item["reorder_level"]', 'return item["stock"] <= item["reorder_level"]'
)

by_rewrite = json.dumps({"path": "inventory.py", "content": fixed_source})
by_edit = json.dumps({
    "path": "inventory.py",
    "old": 'return item["stock"] < item["reorder_level"]',
    "new": 'return item["stock"] <= item["reorder_level"]',
})

print(f"Rewrite the file:  {len(by_rewrite):>5,} characters to generate (about {len(by_rewrite) // 4:,} tokens)")
print(f"Search and replace: {len(by_edit):>5,} characters to generate (about {len(by_edit) // 4:,} tokens)")
print(f"\nThe edit is {len(by_rewrite) / len(by_edit):.0f}x smaller.")

Rewrite the file:  1,531 characters to generate (about 382 tokens)
Search and replace:   143 characters to generate (about 35 tokens)

The edit is 11x smaller.


And this file is tiny. The edit stays the same size whether the file has 50 lines or 5,000, but the rewrite grows with the file. On real codebases the gap is easily 100x, and output tokens usually cost several times more than input tokens and are generated one at a time. That's the speed and the bill.

## 3. Reading with line numbers

Two changes to `read_file` from notebook 05:

1. **Line numbers**, so the model and the human reading the trace can both point at a location.
2. **Ranges**, so the agent can read lines 20 to 25 instead of the whole file. On large files this matters as much as precise editing.

In [5]:
def safe_path(path):
    full = (workspace / path).resolve()
    if not full.is_relative_to(workspace):
        raise PermissionError(f"{path!r} is outside the workspace")
    return full


def read_file(path, start=1, end=None):
    """Show a file, or part of one, with line numbers."""
    lines = safe_path(path).read_text().splitlines()
    end = len(lines) if end is None else min(end, len(lines))
    if not 1 <= start <= max(len(lines), 1):
        return f"Error: start must be between 1 and {len(lines)}"
    shown = "\n".join(f"{n:>4} | {line}" for n, line in enumerate(lines[start - 1 : end], start))
    return f"{path}: lines {start}-{end} of {len(lines)}\n{shown}"


print(read_file("inventory.py", 20, 23))

inventory.py: lines 20-23 of 50
  20 | def needs_restock(item):
  21 |     """True when stock has fallen to the reorder level or below."""
  22 |     return item["stock"] < item["reorder_level"]
  23 | 


## 4. Finding things

Reading files one by one to find a function is how a lost human works. `find` returns every matching line with its file and line number, and the agent reads only around the hits. It's a plain substring search, which is enough to get started. Part 3 gives the agent proper code search.

In [6]:
def find(pattern):
    """Return every line in the project's Python files that contains pattern."""
    hits = []
    for file in sorted(workspace.rglob("*.py")):
        for number, line in enumerate(file.read_text().splitlines(), 1):
            if pattern in line:
                hits.append(f"{file.relative_to(workspace).as_posix()}:{number}: {line.strip()}")
    if not hits:
        return f"No matches for {pattern!r}"
    more = f"\n... and {len(hits) - 50} more" if len(hits) > 50 else ""
    return "\n".join(hits[:50]) + more


print(find("needs_restock"))

inventory.py:20: def needs_restock(item):
inventory.py:32: return sorted(item["name"] for item in items if needs_restock(item))
inventory.py:44: flag = "  RESTOCK" if needs_restock(item) else ""
test_inventory.py:3: from inventory import apply_sale, find_item, low_stock, needs_restock, total_value
test_inventory.py:14: self.assertTrue(needs_restock(ITEMS[0]))
test_inventory.py:17: self.assertTrue(needs_restock(ITEMS[1]))


## 5. The edit tool, and the one rule that matters

`edit_file(path, old, new)` replaces `old` with `new`. Everything rests on one rule:

> **`old` must appear in the file exactly once.**

If it appears twice, which one did the model mean? If you guess, you'll sometimes change the wrong one without anyone noticing. So the tool refuses and asks for more surrounding context. If it appears zero times, the model's picture of the file is wrong, and saying so is far better than doing nothing quietly.

The rest of the function exists to make failures *recoverable*. Every error says what went wrong and what to try next, because the model will read it.

In [7]:
LINE_NUMBER_PREFIX = re.compile(r"^\s*\d+ \| ", re.MULTILINE)


def closest_line_hint(text, old):
    """Point at the line that most resembles what the model was looking for."""
    first = old.strip().splitlines()[0].strip() if old.strip() else ""
    lines = text.splitlines()
    match = difflib.get_close_matches(first, [line.strip() for line in lines], n=1, cutoff=0.6)
    if not match:
        return ""
    number = next(n for n, line in enumerate(lines, 1) if line.strip() == match[0])
    return f" The closest line is {number}: {lines[number - 1]!r}. Check spaces, tabs and quotes."


def edit_file(path, old, new):
    """Replace the single occurrence of old with new."""
    file = safe_path(path)
    text = file.read_text()

    if old == new:
        return "Error: old and new are identical, so there is nothing to change."
    if LINE_NUMBER_PREFIX.search(old):
        return ("Error: old contains line numbers like '  12 | '. They come from read_file's display. "
                "Copy only the code after the bar.")

    count = text.count(old)
    if count == 0:
        return "Error: old was not found in the file." + closest_line_hint(text, old)
    if count > 1:
        lines = [text[: m.start()].count("\n") + 1 for m in re.finditer(re.escape(old), text)]
        return (f"Error: old appears {count} times (lines {lines}). "
                "Include more of the surrounding lines so it matches exactly once.")

    line = text[: text.index(old)].count("\n") + 1
    file.write_text(text.replace(old, new, 1))

    # Show the result, so the model can check its own work without another call.
    changed = new.count("\n") + 1
    return f"Edited {path}.\n" + read_file(path, max(1, line - 2), line + changed + 1)

Now let's try to break it, using a scratch file. No model is involved, so these checks behave the same in every mode.

In [8]:
(workspace / "scratch.py").write_text(
    'def greet(name):\n    return "Hello, " + name\n\n\ndef shout(name):\n    return "HELLO, " + name.upper()\n'
)

cases = {
    "appears twice":   ("return", "yield"),
    "not there":       ("return 'Hello, ' + name", "return f'Hello, {name}'"),
    "copied numbers":  ('   2 |     return "Hello, " + name', "x"),
    "no change":       ("def greet", "def greet"),
    "a good edit":     ('return "Hello, " + name', 'return f"Hello, {name}"'),
}

results = {}
for label, (old, new) in cases.items():
    results[label] = edit_file("scratch.py", old, new)
    print(f"--- {label}\n{results[label]}\n")

--- appears twice
Error: old appears 2 times (lines [2, 6]). Include more of the surrounding lines so it matches exactly once.

--- not there
Error: old was not found in the file. The closest line is 2: '    return "Hello, " + name'. Check spaces, tabs and quotes.

--- copied numbers
Error: old contains line numbers like '  12 | '. They come from read_file's display. Copy only the code after the bar.

--- no change
Error: old and new are identical, so there is nothing to change.

--- a good edit
Edited scratch.py.
scratch.py: lines 1-4 of 6
   1 | def greet(name):
   2 |     return f"Hello, {name}"
   3 | 
   4 | 



In [9]:
# Checkpoint: each failure is refused with a useful message, and only the good edit lands.
assert "appears 2 times" in results["appears twice"]
assert "closest line is 2" in results["not there"]
assert "line numbers" in results["copied numbers"]
assert "identical" in results["no change"]
assert 'return f"Hello, {name}"' in (workspace / "scratch.py").read_text()
(workspace / "scratch.py").unlink()
print("Checkpoint passed.")

Checkpoint passed.


Look at the "not there" case. The model used single quotes where the file has double quotes, which is one of the most common ways edits fail. A bare "not found" leaves it guessing. The hint shows the real line, so it can fix its next attempt on the first try.

The "copied numbers" case looks paranoid, but it happens a lot with weaker models. They see `   2 |     return ...` in `read_file`'s output and copy the whole thing, line number included.

## 6. The agent, now with precise tools

The loop is the same as in notebook 05. The tool list is new, and the system prompt tells the agent how to use it. We also count how many characters the model writes and reads.

In [10]:
def run_tests():
    env = {name: os.environ[name] for name in ("PATH", "SYSTEMROOT") if name in os.environ}
    try:
        done = subprocess.run(
            [sys.executable, "-B", "-m", "unittest", "discover"],
            cwd=workspace, env=env, capture_output=True, text=True, timeout=30,
        )
    except subprocess.TimeoutExpired:
        return "FAILED: the tests took longer than 30 seconds."
    output = done.stderr.replace(str(workspace), ".")[-2000:]
    return ("PASSED\n" if done.returncode == 0 else "FAILED\n") + output


def tool(name, description, **parameters):
    """Build a tool definition. Every parameter is a required string unless it says otherwise."""
    properties = {key: value if isinstance(value, dict) else {"type": "string", "description": value}
                  for key, value in parameters.items()}
    required = [key for key, value in parameters.items() if not isinstance(value, dict) or not value.get("optional")]
    for value in properties.values():
        value.pop("optional", None)
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": properties, "required": required},
    }}


TOOLS = [
    tool("run_tests", "Run the project's unit tests. Output starts with PASSED or FAILED."),
    tool("find", "Search the project's Python files for a piece of text. Returns file:line: text for each match.",
         pattern="The exact text to look for"),
    tool("read_file", "Read a file with line numbers. Use start and end to read only part of it.",
         path="Path relative to the project root",
         start={"type": "integer", "description": "First line to show (default 1)", "optional": True},
         end={"type": "integer", "description": "Last line to show (default: end of file)", "optional": True}),
    tool("edit_file", "Replace text in a file. old must match the file exactly, including indentation, "
                      "and appear exactly once. Don't include line numbers.",
         path="Path relative to the project root", old="The exact text to replace", new="The new text"),
]

REGISTRY = {"run_tests": run_tests, "find": find, "read_file": read_file, "edit_file": edit_file}

print(json.dumps(TOOLS[2], indent=2))

{
  "type": "function",
  "function": {
    "name": "read_file",
    "description": "Read a file with line numbers. Use start and end to read only part of it.",
    "parameters": {
      "type": "object",
      "properties": {
        "path": {
          "type": "string",
          "description": "Path relative to the project root"
        },
        "start": {
          "type": "integer",
          "description": "First line to show (default 1)"
        },
        "end": {
          "type": "integer",
          "description": "Last line to show (default: end of file)"
        }
      },
      "required": [
        "path"
      ]
    }
  }
}


In [11]:
SYSTEM_PROMPT = """You are a careful coding agent working in a small Python project.

1. Run the tests to see what's failing.
2. Use find to locate the code involved, then read only the lines around it with read_file.
3. Fix the code with edit_file. Change as little as possible, and don't edit the tests.
4. Run the tests again. If they still fail, keep going.

Finish with a short summary. Only say the tests pass if you saw them pass."""


def call_model(messages):
    payload = {"model": llm.model, "messages": messages, "tools": TOOLS}
    response = requests.post(url, headers=headers, json=payload, timeout=llm.timeout)
    response.raise_for_status()
    return response.json()


def run_tool_call(call):
    name = call["function"]["name"]
    if name not in REGISTRY:
        return f"Error: unknown tool {name!r}. Available tools: {sorted(REGISTRY)}"
    try:
        arguments = json.loads(call["function"]["arguments"] or "{}")
        return str(REGISTRY[name](**arguments))
    except Exception as error:
        return f"Error: {type(error).__name__}: {error}"


def run_agent(task, max_steps=15):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": task}]
    stats = {"model_calls": 0, "tool_calls": 0, "chars_written": 0, "chars_read": 0, "input_tokens": 0}

    for step in range(1, max_steps + 1):
        data = call_model(messages)
        message = data["choices"][0]["message"]
        messages.append(message)
        stats["model_calls"] += 1
        stats["input_tokens"] += (data.get("usage") or {}).get("prompt_tokens", 0)

        calls = message.get("tool_calls") or []
        if not calls:
            return {"status": "done", "summary": message.get("content"), **stats}

        for call in calls:
            result = run_tool_call(call)
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
            stats["tool_calls"] += 1
            stats["chars_written"] += len(call["function"]["arguments"] or "")
            if call["function"]["name"] in ("find", "read_file"):
                stats["chars_read"] += len(result)
            print(f"[{step}] {call['function']['name']}({call['function']['arguments'][:70]})")
            print(f"    {result.splitlines()[0][:70] if result else '(empty)'}")

    return {"status": "max_steps", "summary": None, **stats}

## 7. Run it

In [12]:
before = run_tests()
assert before.startswith("FAILED")

started = time.perf_counter()
result = run_agent("Two tests are failing. Find the bug and fix it.")
elapsed = time.perf_counter() - started

print()
print("Status: ", result["status"])
print("Summary:", result["summary"])

[1] run_tests({})
    FAILED


[2] find({"pattern": "def needs_restock"})
    inventory.py:20: def needs_restock(item):


[3] read_file({"path": "inventory.py", "start": 20, "end": 23})
    inventory.py: lines 20-23 of 50


[4] edit_file({"path": "inventory.py", "old": "return item[\"stock\"] < item[\"reord)
    Edited inventory.py.


[5] run_tests({})
    PASSED



Status:  done
Summary: Fixed needs_restock in inventory.py. It used < where the docstring says 'or below', so items exactly at the reorder level were missed. All tests pass now.


In [13]:
after = run_tests()
passed = after.startswith("PASSED")

print("What changed:\n")
for line in difflib.unified_diff(
    INVENTORY.splitlines(), (workspace / "inventory.py").read_text().splitlines(),
    "inventory.py (before)", "inventory.py (after)", lineterm="", n=1,
):
    print(line)

print("\n--- Scorecard ---")
print(f"Tests passing:           {passed}")
print(f"Tests left alone:        {(workspace / 'test_inventory.py').read_text() == TESTS}")
print(f"Model calls:             {result['model_calls']}")
print(f"Characters the model wrote (all tool arguments): {result['chars_written']:,}")
print(f"Characters it read from files (find, read_file): {result['chars_read']:,}")
print(f"For comparison, the whole inventory.py:          {len(INVENTORY):,}")
print(f"Time: {elapsed:.1f}s" + (" (simulated)" if llm.offline else ""))

What changed:

--- inventory.py (before)
+++ inventory.py (after)
@@ -21,3 +21,3 @@
     """True when stock has fallen to the reorder level or below."""
-    return item["stock"] < item["reorder_level"]
+    return item["stock"] <= item["reorder_level"]
 

--- Scorecard ---
Tests passing:           True
Tests left alone:        True
Model calls:             6
Characters the model wrote (all tool arguments): 227
Characters it read from files (find, read_file): 243
For comparison, the whole inventory.py:          1,360
Time: 2.8s (simulated)


In [14]:
# Checkpoint: the offline script makes exactly one small edit.
if llm.offline:
    assert passed
    assert result["chars_written"] < len(INVENTORY) / 2, "A precise fix should write far less than the file."
print("Checkpoint passed." if passed else "Not solved this time. Read the trace above to see where it went wrong.")

Checkpoint passed.


**What a real model did.** On Gemma 4 E2B (a 3.6 GB model running locally through Ollama), the agent fixed the bug, but it took 10 model calls instead of 6. The failing test was called `test_low_stock_names`, so it looked at `low_stock` first, then followed it to `needs_restock`, and then read the same three lines twice before editing. It got there, and you can see exactly where a stronger model or a sharper prompt would save calls.

With a real model, read the trace above closely. Did it use `find` first, or read the whole file anyway? Did any `edit_file` call fail, and did it recover from the error message? Weaker models often need one or two tries with `old`, and that's exactly what the hints are for.

## Recap

- Rewriting whole files is slow, expensive and risky. Precise edits only cost what they change.
- The rule that makes search-and-replace safe: `old` must match **exactly once**.
- Line numbers and ranges let the agent read only what it needs.
- Every error message is guidance for the model's next attempt, so write each one with that in mind.

## Exercises

1. Add a `replace_all=False` option to `edit_file` for renames. Why should it be opt-in rather than the default?
2. Make `edit_file` refuse any edit that leaves the file as invalid Python, using `ast.parse`. Return the syntax error to the model.
3. Track which files the agent has read, and refuse to edit a file it hasn't looked at. What kind of mistake does this prevent?

## Clean up

In [15]:
shutil.rmtree(workspace)
print("Removed the temporary workspace.")

Removed the temporary workspace.


**Next:** [07. A Shell Tool](07_shell_tool.ipynb). The most powerful tool we'll ever give the agent, and the most dangerous.